In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path(r"D:\Shikha\bias_project")
EMB = BASE_DIR / "outputs" / "embeddings"

# Each method: (file, score column). All use Garg pronoun pairs, Career-Family,
# period-split, sign convention positive = male->career.
methods = {
    "word2vec": (EMB / "word2vec" / "weat6_scores_word2vec_v2.csv", "weat6_pairs_d"),
    "glove":    (EMB / "glove"    / "weat6_scores_glove_v2.csv",     "weat6_d"),
    "bert":     (EMB / "bert"     / "seat_scores_bert_v2.csv",       "seat_career_family"),
    "roberta":  (EMB / "roberta"  / "seat_scores_roberta.csv",       "seat_career_family"),
}

# Load each, keep (state, period, score), z-score WITHIN method (mean 0, sd 1)
frames = []
for name, (path, col) in methods.items():
    df = pd.read_csv(path)[["state", "period", col]].copy()
    df = df.rename(columns={col: "raw"})
    df["z"] = (df["raw"] - df["raw"].mean()) / df["raw"].std(ddof=1)
    df["method"] = name
    frames.append(df)
    print(f"{name:<9}: {len(df):>3} cells | raw mean {df['raw'].mean():+.4f} sd {df['raw'].std():.4f}")

long = pd.concat(frames, ignore_index=True)

# Pivot to one row per (state, period), z-scores as columns
wide = long.pivot_table(index=["state", "period"], columns="method", values="z")
method_cols = [m for m in methods if m in wide.columns]

# Composite = mean of available method z-scores; require >= 2 methods
wide["n_methods"] = wide[method_cols].notna().sum(axis=1)
wide["composite_z"] = wide[method_cols].mean(axis=1, skipna=True)
wide.loc[wide["n_methods"] < 2, "composite_z"] = np.nan

wide = wide.reset_index()
valid = wide[wide["n_methods"] >= 2].copy()

print(f"\n=== Composite (>= 2 methods) ===")
print(f"Cells with composite : {len(valid)} of {len(wide)} total")
print(f"n_methods distribution:")
print(wide["n_methods"].value_counts().sort_index().to_string())

print(f"\nBy period:")
for p in ["hist_1920s", "modern_2020", "modern_2024"]:
    sub = valid[valid["period"] == p]
    if len(sub):
        print(f"  {p:<13}: {len(sub):>3} cells | mean composite_z = {sub['composite_z'].mean():+.3f}")

OUT = BASE_DIR / "outputs" / "composite"
OUT.mkdir(parents=True, exist_ok=True)
wide.to_csv(OUT / "composite_scores.csv", index=False)
print(f"\nSaved: {OUT / 'composite_scores.csv'}")
print(wide[["state","period","n_methods","composite_z"]].head(12).to_string(index=False))

word2vec :  38 cells | raw mean +0.0036 sd 0.3737
glove    :  81 cells | raw mean +0.7056 sd 0.4329
bert     : 108 cells | raw mean +0.0066 sd 0.0188
roberta  : 108 cells | raw mean +0.0010 sd 0.0046

=== Composite (>= 2 methods) ===
Cells with composite : 108 of 108 total
n_methods distribution:
n_methods
2    27
3    43
4    38

By period:
  hist_1920s   :   9 cells | mean composite_z = -0.195
  modern_2020  :  50 cells | mean composite_z = -0.120
  modern_2024  :  49 cells | mean composite_z = +0.217

Saved: D:\Shikha\bias_project\outputs\composite\composite_scores.csv
state      period  n_methods  composite_z
   AK modern_2020          3     0.004874
   AK modern_2024          3     0.253554
   AL modern_2020          4    -0.551986
   AL modern_2024          3    -0.416859
   AR modern_2020          4    -0.256229
   AR modern_2024          3    -0.295646
   AZ modern_2020          2    -0.612202
   AZ modern_2024          3     0.952480
   CA  hist_1920s          4     0.551510
 